# UdaPlay — Part 2: Research Agent

This notebook implements a stateful research agent on top of the Part 1 RAG store.

**What's defined inline below** (so a reviewer can read the implementation):

1. **Three tools** decorated with `@tool` from `lib.tooling`:
   - `retrieve_game` — vector-store search over the local games catalog
   - `evaluate_retrieval` — LLM-as-judge that scores whether the retrieved docs answer the question
   - `game_web_search` — Tavily-backed fallback for current-events / out-of-catalog questions
2. An **`Agent`** from `lib.agents` orchestrating `START → REASON → (TOOL_CALL → REASON)* → ANSWER → END`
3. A **multi-turn session demo** with pronoun follow-ups that proves the agent retains conversation history across calls.

A small compatibility shim (section 5) probes for the right method names so the notebook works against any reasonable `Agent` API (`invoke` / `run` / `ask` / callable).


## 1. Environment & dependencies

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY missing"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY missing"
print("Keys loaded.")

Keys loaded.


## 2. Connect to the persisted vector store from Part 1

In [ ]:
from lib.vectorstore import GameVectorStore

store = GameVectorStore(persist_dir="chromadb_store", collection_name="games")
print("Collection size:", store.count())

# Self-heal if Part 1 hasn't been run yet.
if store.count() == 0:
    store.load_from_json("games/games.json")
    print("Re-indexed. New size:", store.count())

Collection size: 18


## 3. Define the three tools inline with `@tool`

The `@tool` decorator (from `lib.tooling`) wraps a plain function and exposes the OpenAI tool schema the Agent needs. Each function stays directly callable so we can sanity-check it without spinning up the agent.


### 3a. `retrieve_game` — semantic search over the local catalog

In [ ]:
from lib.tooling import tool

@tool
def retrieve_game(query: str, n_results: int = 5) -> dict:
    """Search the internal game knowledge base for information about a game.

    Use this FIRST for any factual question about a video game, studio,
    platform, or release year. Returns the top-N catalog matches with their
    similarity scores so the model can judge confidence.
    """
    hits = store.query(query, n_results=n_results)
    return {
        "query": query,
        "results": [
            {
                "id": h["id"],
                "document": h["document"],
                "metadata": h["metadata"],
                "similarity": h["similarity"],
            }
            for h in hits
        ],
        "top_similarity": hits[0]["similarity"] if hits else None,
    }

# Sanity check — call it directly:
preview = retrieve_game(query="Elden Ring", n_results=2)
print("Top match:", preview["results"][0]["id"], "-",
      round(preview["top_similarity"], 3))

Top match: elden-ring - 0.741


### 3b. `evaluate_retrieval` — LLM-as-judge

In [ ]:
import json
from openai import OpenAI

_judge_client = OpenAI()  # reused for the judge call

JUDGE_SYSTEM = (
    "You are a strict judge of whether a set of retrieved documents is "
    "sufficient to answer a user's question about video games. Respond with "
    "JSON matching this schema: "
    '{"useful": bool, "confidence": float in [0,1], "reason": string, '
    '"suggested_web_query": string or null}. '
    "Be conservative: if the docs only partially match, or refer to a "
    "different game / year / platform than the question, mark useful=false "
    "and propose a refined web query."
)

@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> dict:
    """Judge whether the retrieved documents answer the user's question.

    Call this AFTER `retrieve_game`. Pass the original question and the list
    of `document` strings you saw in the retrieval result. Returns
    {useful, confidence, reason, suggested_web_query}. Treat useful=false
    as the signal to escalate to `game_web_search`.
    """
    if not retrieved_docs:
        return {
            "useful": False,
            "confidence": 0.0,
            "reason": "No documents were retrieved.",
            "suggested_web_query": question,
        }

    docs_block = "\n\n".join(
        f"[Doc {i+1}]\n{d}" for i, d in enumerate(retrieved_docs)
    )
    user_prompt = (
        f"Question: {question}\n\n"
        f"Retrieved documents:\n{docs_block}\n\n"
        "Return the JSON object now."
    )
    resp = _judge_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": user_prompt},
        ],
        response_format={"type": "json_object"},
        temperature=0,
    )
    return json.loads(resp.choices[0].message.content)

# Sanity check the judge on the preview retrieval above.
judge_demo = evaluate_retrieval(
    question="Which studio made Elden Ring?",
    retrieved_docs=[r["document"] for r in preview["results"]],
)
print(judge_demo)

{'useful': False, 'confidence': 0.8, 'reason': 'The first document provides the correct information about the studio that made Elden Ring, but the second document is irrelevant as it discusses a different game. This partial match is not sufficient to fully answer the question.', 'suggested_web_query': 'Elden Ring developer studio'}


### 3c. `game_web_search` — Tavily fallback

In [ ]:
import os
from tavily import TavilyClient

@tool
def game_web_search(query: str, max_results: int = 5) -> dict:
    """Search the live web (Tavily) when the local catalog is insufficient.

    Use this only if `evaluate_retrieval` returned useful=false OR the
    question is about current / breaking news (e.g. "what is X working on
    right now"). Returns Tavily's answer plus the top result snippets and
    URLs for citation.
    """
    client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
    raw = client.search(
        query=query,
        max_results=max_results,
        search_depth="advanced",
        include_answer=True,
    )
    return {
        "query": query,
        "answer": raw.get("answer"),
        "results": [
            {
                "title": r.get("title"),
                "url": r.get("url"),
                "content": r.get("content"),
                "score": r.get("score"),
            }
            for r in raw.get("results", [])
        ],
    }

/Users/yong.qiang.tan/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 4. Construct the `Agent`

We pass the three `@tool` functions to `lib.agents.Agent`. The system prompt enforces the rubric workflow: **retrieve → evaluate → (web fallback if useful=false or current-events) → answer with citations**.


In [ ]:
from lib.agents import Agent

SYSTEM_PROMPT = """You are UdaPlay, a research assistant for video-game questions.

WORKFLOW (follow strictly):
1. For any factual question about a game, ALWAYS call `retrieve_game` first.
2. Then call `evaluate_retrieval`, passing the original question and the list
   of `document` strings you just saw.
3. If `useful` is false, OR the question is about current / breaking news
   (e.g. "what is X working on right now"), call `game_web_search`.
4. Once you have enough information, produce a final answer.

ANSWER FORMAT:
- Begin with a direct one-sentence answer.
- Add supporting detail (release year, platforms, studios) where relevant.
- End with a "Sources:" line listing either the internal catalog (with game ids)
  or the web URLs you used.
- If you genuinely cannot answer, say so plainly — do not invent facts.

MEMORY:
- When the user asks a follow-up that uses a pronoun ("it", "they", "its
  publisher"), resolve the pronoun against earlier turns in this session
  before deciding which tool to call.
"""

client = OpenAI()

def build_agent(tools, instructions, client, model="gpt-4o-mini", max_iters=6):
    """Defensively construct Agent across common ctor signatures."""
    candidate_kwargs = [
        dict(model_name=model, instructions=instructions, tools=tools,
             openai_client=client, max_iters=max_iters),
        dict(model_name=model, instructions=instructions, tools=tools,
             openai_client=client),
        dict(model=model, instructions=instructions, tools=tools,
             openai_client=client),
        dict(model=model, instructions=instructions, tools=tools, client=client),
        dict(model=model, system_prompt=instructions, tools=tools, client=client),
        dict(model=model, tools=tools, instructions=instructions),
    ]
    last_err = None
    for kw in candidate_kwargs:
        try:
            return Agent(**kw)
        except TypeError as e:
            last_err = e
    raise last_err

agent = build_agent(
    tools=[retrieve_game, evaluate_retrieval, game_web_search],
    instructions=SYSTEM_PROMPT,
    client=client,
)
print("Agent ready:", type(agent).__name__)

Agent ready: Agent


## 5. Compatibility shim — invoke / report / history

The Udacity workspace may name the entry-point method `invoke`, `run`, `ask`, or just `__call__`. The helpers below probe each shape and normalize the result so the cells below are framework-agnostic.


In [ ]:
def invoke_agent(agent, query, session_id=None):
    """Call the agent regardless of which entry-point the framework exposes."""
    attempts = []
    for name in ("invoke", "run", "ask", "chat"):
        if hasattr(agent, name):
            attempts.append((getattr(agent, name), True))   # with session_id
            attempts.append((getattr(agent, name), False))  # without
    if callable(agent):
        attempts.append((agent, True))
        attempts.append((agent, False))

    last_err = None
    for fn, with_session in attempts:
        try:
            if with_session and session_id is not None:
                return fn(query, session_id=session_id)
            return fn(query)
        except TypeError as e:
            last_err = e
            continue
    raise RuntimeError(f"Could not invoke the agent. Last error: {last_err}")


def final_answer(run):
    if isinstance(run, str):
        return run
    for attr in ("final_answer", "output", "answer", "content", "response", "result"):
        v = getattr(run, attr, None)
        if isinstance(v, str) and v.strip():
            return v
    if isinstance(run, dict):
        for k in ("final_answer", "output", "answer", "content", "response", "result"):
            v = run.get(k)
            if isinstance(v, str) and v.strip():
                return v
    return str(run)


def tool_calls(run):
    calls = getattr(run, "tool_calls", None)
    if calls is None and isinstance(run, dict):
        calls = run.get("tool_calls")
    if not calls and hasattr(run, "steps"):
        calls = [getattr(s, "payload", {}) for s in run.steps
                 if getattr(s, "state", None) and "TOOL" in str(s.state)]
    return calls or []


def report(run, query):
    lines = [f"## Question\n{query}", "", "## Answer", final_answer(run),
             "", "## Reasoning trace"]
    calls = tool_calls(run)
    if not calls:
        lines.append("(no tool calls recorded)")
    for i, c in enumerate(calls, 1):
        if isinstance(c, dict):
            name = c.get("name", "?")
            args = c.get("arguments", c.get("args", {}))
        else:
            name = getattr(c, "name", "?")
            args = getattr(c, "arguments", {})
        preview = json.dumps(args, default=str)[:200]
        lines.append(f"{i}. **{name}** — args: `{preview}`")
    return "\n".join(lines)


def session_history(agent, session_id):
    for attr in ("get_history", "session_history", "history"):
        v = getattr(agent, attr, None)
        if callable(v):
            try:
                return v(session_id)
            except TypeError:
                pass
        elif isinstance(v, dict):
            return v.get(session_id, [])
    sessions = getattr(agent, "_sessions", None) or getattr(agent, "sessions", None)
    if isinstance(sessions, dict):
        s = sessions.get(session_id)
        if isinstance(s, list):
            return s
        if hasattr(s, "messages"):
            return s.messages
    return []

## 6. Single-turn example queries

Each call uses `session_id=None`, so the agent treats them as independent turns — a smoke test that each tool path works end-to-end.


In [8]:
q = "Who developed FIFA 21?"
run = invoke_agent(agent, q)
print(report(run, q))


## Question
Who developed FIFA 21?

## Answer
FIFA 21 was developed by EA Vancouver and EA Romania. The game was released in 2020 and is available on platforms such as PlayStation 4, Xbox One, Nintendo Switch, PC, PlayStation 5, and Xbox Series X/S. 

Sources: internal catalog (game id: fifa-21)

## Reasoning trace
1. **retrieve_game** — args: `{"query": "FIFA 21"}`
2. **evaluate_retrieval** — args: `{"question": "Who developed FIFA 21?", "retrieved_docs": [{"document": "FIFA 21 (2020) \u2014 Genre: Sports. Platforms: PlayStation 4, Xbox One, Nintendo Switch, PC, PlayStation 5, Xbox Series X/S. De`


In [ ]:
q = "When was God of War Ragnarok released?"
run = invoke_agent(agent, q)
print(report(run, q))

## Question
When was God of War Ragnarok released?

## Answer
God of War Ragnarok was released in 2022. It is an action-adventure game developed by Santa Monica Studio and is available on PlayStation 4, PlayStation 5, and PC. 

Sources: internal catalog (game id: god-of-war-ragnarok)

## Reasoning trace
1. **retrieve_game** — args: `{"query": "God of War Ragnarok"}`
2. **evaluate_retrieval** — args: `{"question": "When was God of War Ragnarok released?", "retrieved_docs": [{"document": "God of War Ragnarok (2022) \u2014 Genre: Action-Adventure. Platforms: PlayStation 4, PlayStation 5, PC. Develope`


In [ ]:
q = "What platform was Pokemon Red launched on?"
run = invoke_agent(agent, q)
print(report(run, q))

## Question
What platform was Pokemon Red launched on?

## Answer
Pokemon Red was launched on the Game Boy. It was released in 1996 and developed by Game Freak, published by Nintendo. 

Sources: internal catalog (game id: pokemon-red)

## Reasoning trace
1. **retrieve_game** — args: `{"query": "Pokemon Red"}`
2. **evaluate_retrieval** — args: `{"question": "What platform was Pokemon Red launched on?", "retrieved_docs": [{"document": "Pokemon Red (1996) \u2014 Genre: Role-Playing. Platforms: Game Boy. Developer: Game Freak. Publisher: Ninten`


In [ ]:
# Current-events question — should force a game_web_search fallback.
q = "What is Rockstar Games working on right now?"
run = invoke_agent(agent, q)
print(report(run, q))

## Question
What is Rockstar Games working on right now?

## Answer
Rockstar Games is currently working on several projects, including the highly anticipated Grand Theft Auto VI, set to release in 2025, as well as updates to GTA Online. They are also developing remakes of the Max Payne games and exploring a new medieval-themed game. 

Sources: 
- [Rockstar's 2023 New Games & Announcements](https://www.youtube.com/watch?v=FLUtnPDdaNw)
- [Take-Two Interactive Announcement](https://www.take2games.com/ir/news/rockstar-games-announces-grand-theft-auto-vi-coming-2025)

## Reasoning trace
1. **game_web_search** — args: `{"query": "Rockstar Games current projects 2023", "max_results": 5}`


## 7. Multi-turn session — proving conversational memory

Now every call is pinned to the same `session_id`. The follow-ups use **pronouns** (`it`, `its`) that can only resolve against earlier turns the agent remembers.


In [ ]:
SESSION = "demo-session-1"

# Turn 1 — establish the subject.
q1 = "Which studio made Elden Ring?"
turn1 = invoke_agent(agent, q1, session_id=SESSION)
print(report(turn1, q1))

## Question
Which studio made Elden Ring?

## Answer
Elden Ring was developed by FromSoftware. The game was released in 2022 and is available on platforms such as PlayStation 4, Xbox One, PC, PlayStation 5, and Xbox Series X/S. 

Sources: internal catalog (game id: elden-ring)

## Reasoning trace
1. **retrieve_game** — args: `{"query": "Elden Ring"}`
2. **evaluate_retrieval** — args: `{"question": "Which studio made Elden Ring?", "retrieved_docs": [{"id": "elden-ring", "document": "Elden Ring (2022) \u2014 Genre: Action RPG. Platforms: PlayStation 4, Xbox One, PC, PlayStation 5, Xb`


In [ ]:
# Turn 2 — pronoun follow-up ("it" = Elden Ring).
q2 = "Who published it?"
turn2 = invoke_agent(agent, q2, session_id=SESSION)
print(report(turn2, q2))

## Question
Who published it?

## Answer
Elden Ring was published by Bandai Namco Entertainment. 

Sources: internal catalog (game id: elden-ring)

## Reasoning trace
(no tool calls recorded)


In [ ]:
# Turn 3 — another follow-up against the running subject.
q3 = "And what year did it come out?"
turn3 = invoke_agent(agent, q3, session_id=SESSION)
print(report(turn3, q3))

## Question
And what year did it come out?

## Answer
Elden Ring was released in 2022. 

Sources: internal catalog (game id: elden-ring)

## Reasoning trace
(no tool calls recorded)


In [ ]:
# Inspect the session history to confirm the agent really is carrying state
# across the three turns above.
history = session_history(agent, SESSION)
print(f"Session has {len(history)} messages.")
for m in history:
    if isinstance(m, dict):
        role = m.get("role")
        content = m.get("content")
    else:
        role = getattr(m, "role", "?")
        content = getattr(m, "content", "")
    snippet = (content if isinstance(content, str) else "(tool-call message)")
    snippet = snippet.replace("\n", " ")[:120]
    print(f"  - {role}: {snippet}")

Session has 11 messages.
  - system: You are UdaPlay, a research assistant for video-game questions.  WORKFLOW (follow strictly): 1. For any factual question
  - user: Which studio made Elden Ring?
  - assistant: (tool-call message)
  - tool: {"query": "Elden Ring", "results": [{"id": "elden-ring", "document": "Elden Ring (2022) \u2014 Genre: Action RPG. Platfo
  - assistant: (tool-call message)
  - tool: {"useful": true, "confidence": 1.0, "reason": "The retrieved document clearly states that Elden Ring was developed by Fr
  - assistant: Elden Ring was developed by FromSoftware. The game was released in 2022 and is available on platforms such as PlayStatio
  - user: Who published it?
  - assistant: Elden Ring was published by Bandai Namco Entertainment.   Sources: internal catalog (game id: elden-ring)
  - user: And what year did it come out?
  - assistant: Elden Ring was released in 2022.   Sources: internal catalog (game id: elden-ring)


## 8. Notes on the design

- **Tools defined inline.** Each tool is a plain Python function decorated with `@tool`; the decorator generates the OpenAI tool schema from the type hints + docstring. The reviewer can read every line of `retrieve_game`, `evaluate_retrieval`, and `game_web_search` directly above.
- **LLM-as-judge.** `evaluate_retrieval` runs a separate, low-temperature `gpt-4o-mini` call with `response_format={"type": "json_object"}` so the verdict is a structured `{useful, confidence, reason, suggested_web_query}` dict the agent branches on.
- **Two-tier retrieval.** Local Chroma search first; Tavily only fires when the judge says the local docs are insufficient (or the question is current-events). Keeps the web-search call budget low.
- **Session memory.** `invoke_agent(..., session_id=...)` reuses the same message list across calls, so pronouns ("it", "its publisher") resolve against earlier turns. `session_id=None` is the stateless mode used for the single-turn examples.
- **Framework-agnostic shim.** `invoke_agent`, `report`, `session_history` probe for the method names actually exposed by `lib.agents.Agent`, so the notebook runs cleanly against the course's workspace files even if the API differs from my local scaffolds.
